# Preparação de dados — atualização

Esta célula integra melhorias do script de coleta do IBGE: timeout, `raise_for_status()`, uso de `Path`, criação do diretório de destino, e gravação do CSV com `;` e `utf-8`.

In [34]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import requests
import pandas as pd
from pathlib import Path
from IPython.display import display

# Se o notebook está dentro da pasta notebooks/
PASTA_RAIZ = Path.cwd().resolve().parent

PASTA_RAW = PASTA_RAIZ / "data" / "raw"
PASTA_PROCESSED = PASTA_RAIZ / "data" / "processed"
PASTA_ANALYTICS = PASTA_RAIZ / "data" / "analytics"

PASTA_ANALYTICS.mkdir(parents=True, exist_ok=True)

print("Pasta raiz:", PASTA_RAIZ)
print("Raw:", PASTA_RAW)
print("Processed:", PASTA_PROCESSED)
print("Analytics:", PASTA_ANALYTICS)

Pasta raiz: C:\Dash-research
Raw: C:\Dash-research\data\raw
Processed: C:\Dash-research\data\processed
Analytics: C:\Dash-research\data\analytics


PEGA DADOS IBGE

In [35]:
# Se o notebook estiver dentro da pasta notebooks/
PASTA_RAIZ = Path.cwd().resolve().parent

PASTA_RAW = PASTA_RAIZ / "data" / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)


def coletar_dados_ibge_sidra():
    print("Coletando dados da API SIDRA/IBGE...")

    # Tabela 4714 - População residente
    # n6/all = todos os municípios
    # p/2022 = período 2022
    url = (
        "https://apisidra.ibge.gov.br/values/"
        "t/4714"
        "/n6/all"
        "/v/93"
        "/p/2022"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    dados = response.json()

    # A primeira linha costuma ser cabeçalho
    df = pd.DataFrame(dados[1:])

    print("Colunas retornadas pela API:")
    print(df.columns.tolist())

    # Na API SIDRA, normalmente:
    # D1C = código do município
    # D1N = nome do município
    # V = valor
    df = df.rename(
        columns={
            "D1C": "CO_MUNICIPIO_ESC",
            "D1N": "NO_MUNICIPIO_IBGE",
            "V": "POPULACAO_MUNICIPIO"
        }
    )

    df = df[
        [
            "CO_MUNICIPIO_ESC",
            "NO_MUNICIPIO_IBGE",
            "POPULACAO_MUNICIPIO"
        ]
    ]

    df["CO_MUNICIPIO_ESC"] = pd.to_numeric(
        df["CO_MUNICIPIO_ESC"],
        errors="coerce"
    ).astype("Int64")

    df["POPULACAO_MUNICIPIO"] = (
        df["POPULACAO_MUNICIPIO"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df["POPULACAO_MUNICIPIO"] = pd.to_numeric(
        df["POPULACAO_MUNICIPIO"],
        errors="coerce"
    ).astype("Int64")

    return df


df_ibge = coletar_dados_ibge_sidra()

caminho_salvamento = PASTA_RAW / "ibge_populacao_municipios_2022.csv"

df_ibge.to_csv(
    caminho_salvamento,
    index=False,
    sep=";",
    encoding="utf-8"
)

print(f"Sucesso! {len(df_ibge)} municípios coletados.")
print(f"Arquivo salvo em: {caminho_salvamento}")

display(df_ibge.head())

Coletando dados da API SIDRA/IBGE...
Colunas retornadas pela API:
['NC', 'NN', 'MC', 'MN', 'V', 'D1C', 'D1N', 'D2C', 'D2N', 'D3C', 'D3N']
Sucesso! 5570 municípios coletados.
Arquivo salvo em: C:\Dash-research\data\raw\ibge_populacao_municipios_2022.csv


,CO_MUNICIPIO_ESC,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO
0,1100015,Alta Floresta D'Oeste - RO,21494
1,1100023,Ariquemes - RO,96833
2,1100031,Cabixi - RO,5351
3,1100049,Cacoal - RO,86887
4,1100056,Cerejeiras - RO,15890


Carrega dados do CSV ENEM

In [36]:
caminho_enem =  PASTA_PROCESSED / "enem_2022_tratado.csv"

df_enem = pd.read_csv(
    caminho_enem,
    sep=";",
    encoding="utf-8"
)

print(f"ENEM: {df_enem.shape[0]} linhas e {df_enem.shape[1]} colunas")
display(df_enem.head())

ENEM: 2504014 linhas e 18 colunas


,NU_ANO,SG_UF_PROVA,TP_SEXO,TP_FAIXA_ETARIA,TP_ESCOLA,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,Q006,Q025,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO
0,2022,BA,F,5,1,NaN,NaN,NaN,NaN,2925758,Presidente Tancredo Neves,B,B,421.1,546.0,498.8,565.3,760.0
1,2022,ES,M,6,1,NaN,NaN,NaN,NaN,3201308,Cariacica,A,B,490.7,388.6,357.8,416.0,320.0
2,2022,RJ,F,6,1,NaN,NaN,NaN,NaN,3304904,São Gonçalo,B,B,398.1,427.3,400.2,404.9,440.0
3,2022,PE,F,4,1,NaN,NaN,NaN,NaN,2601201,Arcoverde,B,B,467.5,461.0,466.7,435.3,360.0
4,2022,SE,F,2,3,NaN,NaN,NaN,NaN,2804508,Nossa Senhora da Glória,B,B,458.7,539.8,488.2,456.8,940.0


Carrega IBGE

In [37]:
caminho_ibge = PASTA_RAW / "ibge_populacao_municipios_2022.csv"

df_ibge = pd.read_csv(
    caminho_ibge,
    sep=";",
    encoding="utf-8"
)

print(f"IBGE: {df_ibge.shape[0]} linhas e {df_ibge.shape[1]} colunas")
display(df_ibge.head())

IBGE: 5570 linhas e 3 colunas


,CO_MUNICIPIO_ESC,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO
0,1100015,Alta Floresta D'Oeste - RO,21494
1,1100023,Ariquemes - RO,96833
2,1100031,Cabixi - RO,5351
3,1100049,Cacoal - RO,86887
4,1100056,Cerejeiras - RO,15890


DIAGNOSTICO DE AUSENTES

In [38]:
def diagnosticar_ausentes(df, nome_base):
    print(f"\n===== Valores ausentes — {nome_base} =====")
    
    ausentes = (
        df.isna()
        .sum()
        .reset_index()
    )
    
    ausentes.columns = ["COLUNA", "QTD_AUSENTES"]
    ausentes["PERCENTUAL"] = (ausentes["QTD_AUSENTES"] / len(df) * 100).round(2)
    
    ausentes = ausentes.sort_values("PERCENTUAL", ascending=False)
    
    display(ausentes)
    
    return ausentes


diagnosticos_enem = {}

diagnosticos_enem = diagnosticar_ausentes(df_enem, f"ENEM 2022")

diagnostico_ibge = diagnosticar_ausentes(df_ibge, "IBGE")


===== Valores ausentes — ENEM 2022 =====


,COLUNA,QTD_AUSENTES,PERCENTUAL
8,TP_SIT_FUNC_ESC,1772340,70.78
5,SG_UF_ESC,1772340,70.78
7,TP_LOCALIZACAO_ESC,1772340,70.78
6,TP_DEPENDENCIA_ADM_ESC,1772340,70.78
16,NU_NOTA_MT,148619,5.94
13,NU_NOTA_CN,148619,5.94
17,NU_NOTA_REDACAO,10572,0.42
14,NU_NOTA_CH,10572,0.42
15,NU_NOTA_LC,10572,0.42
0,NU_ANO,0,0.00



===== Valores ausentes — IBGE =====


,COLUNA,QTD_AUSENTES,PERCENTUAL
0,CO_MUNICIPIO_ESC,0,0.0
1,NO_MUNICIPIO_IBGE,0,0.0
2,POPULACAO_MUNICIPIO,0,0.0


Padronização ENEM

In [39]:
COLUNAS_NOTAS = [
    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "NU_NOTA_REDACAO"
]

COLUNAS_CATEGORICAS = [
    "TP_SEXO",
    "TP_ESCOLA",
    "Q006",
    "Q025",
    "SG_UF_PROVA",
    "SG_UF_ESC",
    "TP_DEPENDENCIA_ADM_ESC",
    "TP_LOCALIZACAO_ESC",
    "TP_SIT_FUNC_ESC"
]

COLUNAS_INTEIRAS = [
    "NU_ANO",
    "ANO_BASE",
    "NU_IDADE",
    "CO_MUNICIPIO_ESC"
]


def padronizar_tipos_enem(df):
    df = df.copy()
    
    # Padroniza notas como número
    for coluna in COLUNAS_NOTAS:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    
    # Padroniza colunas inteiras
    for coluna in COLUNAS_INTEIRAS:
        if coluna in df.columns:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")
    
    # Padroniza textos/categorias
    for coluna in COLUNAS_CATEGORICAS:
        if coluna in df.columns:
            df[coluna] = df[coluna].astype("string").str.strip()
    
    return df


bases_enem_padronizadas = {}

bases_enem_padronizadas[2022] = padronizar_tipos_enem(df_enem)
print(f"Tipos padronizados — ENEM 2022")
print(bases_enem_padronizadas[2022].dtypes)

Tipos padronizados — ENEM 2022
NU_ANO                      Int64
SG_UF_PROVA                string
TP_SEXO                    string
TP_FAIXA_ETARIA             int64
TP_ESCOLA                  string
SG_UF_ESC                  string
TP_DEPENDENCIA_ADM_ESC     string
TP_LOCALIZACAO_ESC         string
TP_SIT_FUNC_ESC            string
CO_MUNICIPIO_PROVA          int64
NO_MUNICIPIO_PROVA            str
Q006                       string
Q025                       string
NU_NOTA_CN                float64
NU_NOTA_CH                float64
NU_NOTA_LC                float64
NU_NOTA_MT                float64
NU_NOTA_REDACAO           float64
dtype: object


Remoção de inconsistências do ENEM

Aqui vamos remover:

* linhas sem nenhuma nota;
* notas fora do intervalo esperado;
* idades inválidas;
* registros sem ano.

In [40]:
def limpar_inconsistencias_enem(df):
    df = df.copy()
    
    linhas_iniciais = len(df)
    
    # Remove registros sem ano
    if "NU_ANO" in df.columns:
        df = df.dropna(subset=["NU_ANO"])
    
    # Remove linhas sem nenhuma nota
    colunas_notas_existentes = [col for col in COLUNAS_NOTAS if col in df.columns]
    
    if colunas_notas_existentes:
        df = df.dropna(
            subset=colunas_notas_existentes,
            how="all"
        )
    
    # Remove notas fora dos intervalos esperados
    # Provas objetivas: 0 a 1000
    # Redação: 0 a 1000
    for coluna in colunas_notas_existentes:
        df.loc[
            (df[coluna] < 0) | (df[coluna] > 1000),
            coluna
        ] = pd.NA
    
    # Remove idades inconsistentes
    if "NU_IDADE" in df.columns:
        df.loc[
            (df["NU_IDADE"] < 10) | (df["NU_IDADE"] > 100),
            "NU_IDADE"
        ] = pd.NA
    
    linhas_finais = len(df)
    
    print(f"Linhas iniciais: {linhas_iniciais}")
    print(f"Linhas finais: {linhas_finais}")
    print(f"Linhas removidas: {linhas_iniciais - linhas_finais}")
    
    return df


bases_enem_limpas = {}

print(f"\n===== Limpando ENEM 2022 =====")
bases_enem_limpas[2022] = limpar_inconsistencias_enem(df_enem)


===== Limpando ENEM 2022 =====
Linhas iniciais: 2504014
Linhas finais: 2504014
Linhas removidas: 0


Tratamento de valores ausentes no ENEM

Aqui não vamos “inventar nota”. Para análise, é mais correto:

* manter nota ausente como NaN;
* preencher categorias ausentes como "Não informado";
* calcular média geral usando as notas disponíveis.

In [41]:
def tratar_ausentes_enem(df):
    df = df.copy()
    
    # Categorias que podem receber "Não informado"
    colunas_categoricas_para_preencher = [
        "TP_SEXO",
        "TP_ESCOLA",
        "Q006",
        "Q025",
        "SG_UF_PROVA",
        "SG_UF_ESC",
        "TP_DEPENDENCIA_ADM_ESC",
        "TP_LOCALIZACAO_ESC",
        "TP_SIT_FUNC_ESC"
    ]
    
    for coluna in colunas_categoricas_para_preencher:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna("Não informado")
    
    # Idade: manter ausente, mas criar faixa "Não informado"
    if "NU_IDADE" in df.columns:
        df["NU_IDADE"] = pd.to_numeric(df["NU_IDADE"], errors="coerce")
    
    # Média geral com as notas existentes
    colunas_notas_existentes = [col for col in COLUNAS_NOTAS if col in df.columns]
    
    df["MEDIA_GERAL"] = df[colunas_notas_existentes].mean(axis=1)
    
    # Remove quem ficou sem média geral
    df = df.dropna(subset=["MEDIA_GERAL"])
    
    return df


bases_enem_tratadas = {}

print(f"\n===== Tratando ausentes ENEM 2022 =====")
bases_enem_tratadas[2022] = tratar_ausentes_enem(df_enem)
print(f"Tratamento concluído — ENEM 2022")
print(bases_enem_tratadas[2022].head())


===== Tratando ausentes ENEM 2022 =====
Tratamento concluído — ENEM 2022
   NU_ANO SG_UF_PROVA TP_SEXO  TP_FAIXA_ETARIA  TP_ESCOLA      SG_UF_ESC  \
0    2022          BA       F                5          1  Não informado   
1    2022          ES       M                6          1  Não informado   
2    2022          RJ       F                6          1  Não informado   
3    2022          PE       F                4          1  Não informado   
4    2022          SE       F                2          3  Não informado   

  TP_DEPENDENCIA_ADM_ESC TP_LOCALIZACAO_ESC TP_SIT_FUNC_ESC  \
0          Não informado      Não informado   Não informado   
1          Não informado      Não informado   Não informado   
2          Não informado      Não informado   Não informado   
3          Não informado      Não informado   Não informado   
4          Não informado      Não informado   Não informado   

   CO_MUNICIPIO_PROVA         NO_MUNICIPIO_PROVA Q006 Q025  NU_NOTA_CN  \
0             29

PADRONIZAÇÃO DA TABELA ENEM

In [42]:
MAPA_RENDA = {
    "A": "Nenhuma renda",
    "B": "Até 1 salário mínimo",
    "C": "1 a 1,5 salários",
    "D": "1,5 a 2 salários",
    "E": "2 a 2,5 salários",
    "F": "2,5 a 3 salários",
    "G": "3 a 4 salários",
    "H": "4 a 5 salários",
    "I": "5 a 6 salários",
    "J": "6 a 7 salários",
    "K": "7 a 8 salários",
    "L": "8 a 9 salários",
    "M": "9 a 10 salários",
    "N": "10 a 12 salários",
    "O": "12 a 15 salários",
    "P": "15 a 20 salários",
    "Q": "Acima de 20 salários"
}

MAPA_INTERNET = {
    "A": "Não",
    "B": "Sim"
}

MAPA_TIPO_ESCOLA = {
    "1": "Não respondeu",
    "2": "Pública",
    "3": "Privada",
    1: "Não respondeu",
    2: "Pública",
    3: "Privada"
}

MAPA_DEPENDENCIA = {
    "1": "Federal",
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
    1: "Federal",
    2: "Estadual",
    3: "Municipal",
    4: "Privada"
}

MAPA_LOCALIZACAO = {
    "1": "Urbana",
    "2": "Rural",
    1: "Urbana",
    2: "Rural"
}


def criar_variaveis_analise_enem(df):
    df = df.copy()
    
    if "Q006" in df.columns:
        df["RENDA_FAMILIAR"] = df["Q006"].map(MAPA_RENDA).fillna("Não informado")
    
    if "Q025" in df.columns:
        df["ACESSO_INTERNET"] = df["Q025"].map(MAPA_INTERNET).fillna("Não informado")
    
    if "TP_ESCOLA" in df.columns:
        df["TIPO_ESCOLA"] = df["TP_ESCOLA"].map(MAPA_TIPO_ESCOLA).fillna("Não informado")
    
    if "TP_DEPENDENCIA_ADM_ESC" in df.columns:
        df["DEPENDENCIA_ESCOLA"] = (
            df["TP_DEPENDENCIA_ADM_ESC"]
            .map(MAPA_DEPENDENCIA)
            .fillna("Não informado")
        )
    
    if "TP_LOCALIZACAO_ESC" in df.columns:
        df["LOCALIZACAO_ESCOLA"] = (
            df["TP_LOCALIZACAO_ESC"]
            .map(MAPA_LOCALIZACAO)
            .fillna("Não informado")
        )
    
    if "NU_IDADE" in df.columns:
        df["FAIXA_ETARIA"] = pd.cut(
            df["NU_IDADE"],
            bins=[0, 17, 20, 25, 30, 40, 100],
            labels=[
                "Até 17",
                "18 a 20",
                "21 a 25",
                "26 a 30",
                "31 a 40",
                "Acima de 40"
            ]
        )
        
        df["FAIXA_ETARIA"] = df["FAIXA_ETARIA"].astype("string").fillna("Não informado")
    
    return df


bases_enem_final = {}

print(f"\n===== Criando variáveis de análise ENEM 2022 =====")
bases_enem_final[2022] = criar_variaveis_analise_enem(bases_enem_tratadas[2022])
print(bases_enem_final[2022].head())


===== Criando variáveis de análise ENEM 2022 =====
   NU_ANO SG_UF_PROVA TP_SEXO  TP_FAIXA_ETARIA  TP_ESCOLA      SG_UF_ESC  \
0    2022          BA       F                5          1  Não informado   
1    2022          ES       M                6          1  Não informado   
2    2022          RJ       F                6          1  Não informado   
3    2022          PE       F                4          1  Não informado   
4    2022          SE       F                2          3  Não informado   

  TP_DEPENDENCIA_ADM_ESC TP_LOCALIZACAO_ESC TP_SIT_FUNC_ESC  \
0          Não informado      Não informado   Não informado   
1          Não informado      Não informado   Não informado   
2          Não informado      Não informado   Não informado   
3          Não informado      Não informado   Não informado   
4          Não informado      Não informado   Não informado   

   CO_MUNICIPIO_PROVA  ... NU_NOTA_CH NU_NOTA_LC NU_NOTA_MT  NU_NOTA_REDACAO  \
0             2925758  ...      

TRATAMENTO IBGE

In [48]:
def tratar_ibge(df_ibge):
    df = df_ibge.copy()
    
    print("Linhas iniciais IBGE:", len(df))
    
    # Se a base veio com CO_MUNICIPIO_ESC, renomeia para CO_MUNICIPIO_PROVA
    if "CO_MUNICIPIO_ESC" in df.columns and "CO_MUNICIPIO_PROVA" not in df.columns:
        df = df.rename(columns={"CO_MUNICIPIO_ESC": "CO_MUNICIPIO_PROVA"})
    
    # Padroniza código do município
    if "CO_MUNICIPIO_PROVA" in df.columns:
        df["CO_MUNICIPIO_PROVA"] = pd.to_numeric(
            df["CO_MUNICIPIO_PROVA"],
            errors="coerce"
        ).astype("Int64")
    else:
        raise KeyError("A coluna CO_MUNICIPIO_PROVA não existe na base do IBGE.")
    
    # Padroniza nome do município
    if "NO_MUNICIPIO_IBGE" in df.columns:
        df["NO_MUNICIPIO_IBGE"] = (
            df["NO_MUNICIPIO_IBGE"]
            .astype("string")
            .str.strip()
            .str.title()
        )
    
    # Padroniza população
    if "POPULACAO_MUNICIPIO" in df.columns:
        df["POPULACAO_MUNICIPIO"] = pd.to_numeric(
            df["POPULACAO_MUNICIPIO"],
            errors="coerce"
        )
    else:
        raise KeyError("A coluna POPULACAO_MUNICIPIO não existe na base do IBGE.")
    
    # Remove registros sem chave de município
    df = df.dropna(subset=["CO_MUNICIPIO_PROVA"])
    
    # Remove população ausente ou inválida
    df = df.dropna(subset=["POPULACAO_MUNICIPIO"])
    df = df[df["POPULACAO_MUNICIPIO"] > 0]
    
    # Remove duplicidades por município
    df = df.drop_duplicates(subset=["CO_MUNICIPIO_PROVA"])
    
    # Cria faixa de porte populacional
    df["PORTE_MUNICIPIO"] = pd.cut(
        df["POPULACAO_MUNICIPIO"],
        bins=[0, 20000, 100000, 500000, 1000000, 50000000],
        labels=[
            "Até 20 mil",
            "20 mil a 100 mil",
            "100 mil a 500 mil",
            "500 mil a 1 milhão",
            "Acima de 1 milhão"
        ]
    )
    
    print("Linhas finais IBGE:", len(df))
    
    return df


df_ibge_tratado = tratar_ibge(df_ibge)

display(df_ibge_tratado.head())
display(df_ibge_tratado.dtypes)

Linhas iniciais IBGE: 5570
Linhas finais IBGE: 5570


,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO,PORTE_MUNICIPIO
0,1100015,Alta Floresta D'Oeste - Ro,21494,20 mil a 100 mil
1,1100023,Ariquemes - Ro,96833,20 mil a 100 mil
2,1100031,Cabixi - Ro,5351,Até 20 mil
3,1100049,Cacoal - Ro,86887,20 mil a 100 mil
4,1100056,Cerejeiras - Ro,15890,Até 20 mil


CO_MUNICIPIO_PROVA        Int64
NO_MUNICIPIO_IBGE        string
POPULACAO_MUNICIPIO       int64
PORTE_MUNICIPIO        category
dtype: object

SALVAR DADOS TRATADO

In [50]:
caminho_enem_saida = PASTA_ANALYTICS / f"enem_{2022}_limpo.csv"
    
bases_enem_final[2022].to_csv(
    caminho_enem_saida,
    sep=";",
    encoding="utf-8",
    index=False
)
    
print(f"Base ENEM {2022} limpa salva em: {caminho_enem_saida}")


caminho_ibge_saida = PASTA_ANALYTICS / "ibge_populacao_municipios_2022_limpo.csv"

df_ibge_tratado.to_csv(
    caminho_ibge_saida,
    sep=";",
    encoding="utf-8",
    index=False
)

print(f"Base IBGE limpa salva em: {caminho_ibge_saida}")

Base ENEM 2022 limpa salva em: C:\Dash-research\data\analytics\enem_2022_limpo.csv
Base IBGE limpa salva em: C:\Dash-research\data\analytics\ibge_populacao_municipios_2022_limpo.csv


Merge dos arquivos

In [51]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PASTA_RAIZ = Path.cwd().resolve().parent
PASTA_ANALYTICS = PASTA_RAIZ / "data" / "analytics"

caminho_enem_2022 = PASTA_ANALYTICS / "enem_2022_limpo.csv"
caminho_ibge_2022 = PASTA_ANALYTICS / "ibge_populacao_municipios_2022_limpo.csv"

df_enem_2022 = pd.read_csv(
    caminho_enem_2022,
    sep=";",
    encoding="utf-8"
)

df_ibge_2022 = pd.read_csv(
    caminho_ibge_2022,
    sep=";",
    encoding="utf-8"
)

print("ENEM 2022:", df_enem_2022.shape)
print("IBGE 2022:", df_ibge_2022.shape)

display(df_enem_2022.head())
display(df_ibge_2022.head())



ENEM 2022: (2504014, 24)
IBGE 2022: (5570, 4)


,NU_ANO,SG_UF_PROVA,TP_SEXO,TP_FAIXA_ETARIA,TP_ESCOLA,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,...,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,MEDIA_GERAL,RENDA_FAMILIAR,ACESSO_INTERNET,TIPO_ESCOLA,DEPENDENCIA_ESCOLA,LOCALIZACAO_ESCOLA
0,2022,BA,F,5,1,Não informado,Não informado,Não informado,Não informado,2925758,...,546.0,498.8,565.3,760.0,558.24,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado
1,2022,ES,M,6,1,Não informado,Não informado,Não informado,Não informado,3201308,...,388.6,357.8,416.0,320.0,394.62,Nenhuma renda,Sim,Não respondeu,Não informado,Não informado
2,2022,RJ,F,6,1,Não informado,Não informado,Não informado,Não informado,3304904,...,427.3,400.2,404.9,440.0,414.10,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado
3,2022,PE,F,4,1,Não informado,Não informado,Não informado,Não informado,2601201,...,461.0,466.7,435.3,360.0,438.10,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado
4,2022,SE,F,2,3,Não informado,Não informado,Não informado,Não informado,2804508,...,539.8,488.2,456.8,940.0,576.70,Até 1 salário mínimo,Sim,Privada,Não informado,Não informado


,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO,PORTE_MUNICIPIO
0,1100015,Alta Floresta D'Oeste - Ro,21494,20 mil a 100 mil
1,1100023,Ariquemes - Ro,96833,20 mil a 100 mil
2,1100031,Cabixi - Ro,5351,Até 20 mil
3,1100049,Cacoal - Ro,86887,20 mil a 100 mil
4,1100056,Cerejeiras - Ro,15890,Até 20 mil


In [52]:
if "CO_MUNICIPIO_PROVA" not in df_enem_2022.columns:
    raise KeyError("A coluna CO_MUNICIPIO_PROVA não existe na base do ENEM 2022.")

if "CO_MUNICIPIO_PROVA" not in df_ibge_2022.columns:
    raise KeyError("A coluna CO_MUNICIPIO_PROVA não existe na base do IBGE 2022.")

df_enem_2022["CO_MUNICIPIO_PROVA"] = pd.to_numeric(
    df_enem_2022["CO_MUNICIPIO_PROVA"],
    errors="coerce"
).astype("Int64")

df_ibge_2022["CO_MUNICIPIO_PROVA"] = pd.to_numeric(
    df_ibge_2022["CO_MUNICIPIO_PROVA"],
    errors="coerce"
).astype("Int64")

df_enem_ibge_2022 = pd.merge(
    df_enem_2022,
    df_ibge_2022,
    how="left",
    on="CO_MUNICIPIO_PROVA"
)

print("Base integrada:", df_enem_ibge_2022.shape)

percentual_com_ibge = df_enem_ibge_2022["POPULACAO_MUNICIPIO"].notna().mean() * 100

print(f"Percentual com correspondência no IBGE: {percentual_com_ibge:.2f}%")

display(df_enem_ibge_2022.head())

Base integrada: (2504014, 27)
Percentual com correspondência no IBGE: 100.00%


,NU_ANO,SG_UF_PROVA,TP_SEXO,TP_FAIXA_ETARIA,TP_ESCOLA,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,...,NU_NOTA_REDACAO,MEDIA_GERAL,RENDA_FAMILIAR,ACESSO_INTERNET,TIPO_ESCOLA,DEPENDENCIA_ESCOLA,LOCALIZACAO_ESCOLA,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO,PORTE_MUNICIPIO
0,2022,BA,F,5,1,Não informado,Não informado,Não informado,Não informado,2925758,...,760.0,558.24,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado,Presidente Tancredo Neves - Ba,27734,20 mil a 100 mil
1,2022,ES,M,6,1,Não informado,Não informado,Não informado,Não informado,3201308,...,320.0,394.62,Nenhuma renda,Sim,Não respondeu,Não informado,Não informado,Cariacica - Es,353491,100 mil a 500 mil
2,2022,RJ,F,6,1,Não informado,Não informado,Não informado,Não informado,3304904,...,440.0,414.10,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado,São Gonçalo - Rj,896744,500 mil a 1 milhão
3,2022,PE,F,4,1,Não informado,Não informado,Não informado,Não informado,2601201,...,360.0,438.10,Até 1 salário mínimo,Sim,Não respondeu,Não informado,Não informado,Arcoverde - Pe,77742,20 mil a 100 mil
4,2022,SE,F,2,3,Não informado,Não informado,Não informado,Não informado,2804508,...,940.0,576.70,Até 1 salário mínimo,Sim,Privada,Não informado,Não informado,Nossa Senhora Da Glória - Se,41212,20 mil a 100 mil


In [53]:
from pathlib import Path

PASTA_RAIZ = Path.cwd().resolve().parent
PASTA_ANALYTICS = PASTA_RAIZ / "data" / "analytics"
PASTA_ANALYTICS.mkdir(parents=True, exist_ok=True)

caminho_saida = PASTA_ANALYTICS / "enem_2022_ibge_integrado.csv"

df_enem_ibge_2022.to_csv(
    caminho_saida,
    sep=";",
    encoding="utf-8",
    index=False
)

print(f"Base integrada salva em: {caminho_saida}")
print("Linhas e colunas:", df_enem_ibge_2022.shape)

Base integrada salva em: C:\Dash-research\data\analytics\enem_2022_ibge_integrado.csv
Linhas e colunas: (2504014, 27)
